# Information Extraction & Named Entity Recognition

In den Digital Humanities reicht es oft nicht, dass ein Computer Wörter nur als "Zeichenketten" feststellt. Wir wollen **Information Extraction** betreiben (vgl. Jurafsky/Martin, Kapitel 17) – wir wollen strukturierte historische Fakten aus einem unstrukturierten Briefwechsel ziehen.

**Named Entity Recognition (NER)** (Eigennamenerkennung) ist das wichtigste Werkzeug dafür.

## Das BIO-Format (Sequence Labeling)

Wie erfasst ein Computer überhaupt einen Eigennamen? Für die Maschine ist unser Text keine Einheit, sondern eine Sequenz von Einzelwörtern (Tokens). Die KI muss deshalb jedes einzelne Token taggen (Sequence Labeling).

Der Goldstandard hierfür ist das **BIO-Format**:
- **B (Begin):** Das Token ist der Beginn einer Entität.
- **I (Inside):** Das Token ist die Fortsetzung der Entität.
- **O (Outside):** Das Token gehört zu gar keiner Entität.

### ✍️ Gruppenarbeit 1: Manuelles POS- und BIO-Tagging

Nehmt euch 10 Minuten Zeit und taggt den Satz:
*"Gaius Iulius Caesar copias ad flumen Rubiconem duxit."*

**1. UPOS (Universal Part-of-Speech)**
Welche der folgenden 17 Standard-Tags passen zu welchen Wörtern?
`ADJ` (Adjektiv), `ADP` (Präposition), `ADV` (Adverb), `AUX` (Hilfsverb), `CCONJ` (Koord. Konjunktion), `DET` (Determinierer), `INTJ` (Interjektion), `NOUN` (Nomen), `NUM` (Numerale), `PART` (Partikel), `PRON` (Pronomen), `PROPN` (Eigenname), `PUNCT` (Interpunktion), `SCONJ` (Subord. Konjunktion), `SYM` (Symbol), `VERB` (Verb), `X` (Andere).

*Warum POS?* Ein Wort, das als `PROPN` getaggt wird, ist der Hauptverdächtige für eine Entität!

**2. Entitäten (BIO-Format)**
- Nutzt B-, I- und O-Tags für `PER` (Personen), `LOC` (Orte), `ORG` (Organisationen).

## 0. Die NLP-Pipeline im Code aufbauen
Wir laden nun unser NLP-Modell. Achtung auf die Code-Formatierung.

In [ ]:
import spacy
from spacy import displacy
import pandas as pd
from pathlib import Path
import requests
import matplotlib.pyplot as plt

try:
    nlp = spacy.load("la_core_web_lg")
    print("LatinCy Modell einsatzbereit!")
except OSError:
    print(
        "Bitte vorher ausführen: "
        "!python -m spacy download la_core_web_lg"
    )

Schauen wir uns nun die **Musterlösung** an, wie die KI den Testsatz tatsächlich bewertet:

In [ ]:
satz = "Gaius Iulius Caesar copias ad flumen Rubiconem duxit."
testsatz = nlp(satz)

for tok in testsatz:
    print(
        f"{tok.text:15} -> POS: {tok.pos_:5} "
        f"| NER: {tok.ent_iob_}-{tok.ent_type_}"
    )

## Teil 2: Historische Briefe auswerten
Laden wir nun die echten Cicero-Briefe als Datensatz.

In [ ]:
csv_path = Path("outputs") / "cicero_letters.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    briefe = df.dropna(subset=["text"])
    beispiel = briefe.iloc[0]["text"]
    print(f"{len(briefe)} Briefe bereit.")
else:
    print("Excel/CSV nicht gefunden.")

In [ ]:
# NLP auf einen Brief anwenden
doc = nlp(beispiel[:1000])

# displacy visualisiert die Entitäten:
displacy.render(doc, style="ent", jupyter=True)

## Das Kernproblem: Ambiguität und Entity Linking

**Drei Probleme der Ambiguität:**
1. **Ontologisch:** Ist "Roma" die Göttin, das Römische Reich oder die Stadtgemeinde?
2. **Akteure:** Meint "Caesar" C. Iulius Caesar, einen Verwandten der Gens Iulia, oder den kaiserlichen Titel?
3. **Morphologisch:** "Athenis" (in Athen) und "Athenae" sind das Gleiche, die KI erkennt es oft nicht.

**Die Lösung: Automatisches Entity Linking (Grounding)**
Wir verknüpfen jeden gefundenen Namen mit einem Lexikon wie **Pleiades** (für antike Geografie) oder einer Enzyklopädie wie **Wikipedia**.

### Methode A: Entity Linking mit dem Pleiades-Datensatz
Statt eines kleinen Dummys laden wir nun die offizielle `places.csv` der historischen Pleiades-Datenbank (Information Extraction trifft Data Science). Wir vergleichen die gefundenen Orte in 10 Cicero-Briefen direkt mit dem Datensatz.

In [ ]:
import pandas as pd

# Pleiades Datenbank (16 MB) einlesen
df_places = pd.read_csv("data/places.csv")
df_places = df_places.dropna(subset=['representative_latitude', 'representative_longitude'])

# NLP auf 10 Briefe anwenden, um mehr Orte zu finden
orte_aus_briefen = []
for text in briefe.head(10)["text"]:
    doc_10 = nlp(text)
    for ent in doc_10.ents:
        if ent.label_ in ["LOC", "GPE"]:
            orte_aus_briefen.append(ent.lemma_) # lemma = Grundform

# Duplikate entfernen
orte_aus_briefen = list(set(orte_aus_briefen))

In [ ]:
# Wir durchsuchen das riesige CSV nach unseren Funden
gefundene_orte_fuer_karte = []

for ort in orte_aus_briefen:
    # Wir vergleichen den Text mit der "title" Spalte in Pleiades
    treffer = df_places[df_places["title"].str.lower() == ort.lower()]
    
    if not treffer.empty:
        erste_zeile = treffer.iloc[0]
        gefundene_orte_fuer_karte.append((ort, erste_zeile["representative_latitude"], erste_zeile["representative_longitude"]))
        print(f"✓ {ort} gefixt! Pleiades-ID: {erste_zeile['id']}")
    else:
        print(f"X {ort} -> In Pleiades nicht unter diesem Namen auffindbar.")

### Die Entitäten auf der Karte visualisieren
Dank der Disambiguierung über unseren Pleiades-Katalog haben wir jetzt Koordinaten. Dadurch können wir die historischen Textfunde sofort auf eine (interaktive) Landkarte werfen. Das eröffnet den Einstieg in **Geographic Information Systems (GIS)**.

In [ ]:
import folium
from IPython.display import HTML, display

antike_karte = folium.Map(location=[40.0, 15.0], zoom_start=5)

for ort, lat, lon in gefundene_orte_fuer_karte:
    folium.Marker(
        [lat, lon], 
        popup=f"Erwähnt in Brief: {ort}",
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(antike_karte)

display(HTML(antike_karte._repr_html_()))

### Methode B: Live Entity Linking via API
Hier funkt eine Blackbox-Funktion direkt zum Wikidata-Server.

In [ ]:
import requests
from IPython.display import display, HTML

def auto_link_wikidata(suchbegriff):
    """Sucht live auf Wikidata nach dem Begriff und liefert die Q-ID."""
    url = (
        f"https://www.wikidata.org/w/api.php?action=wbsearchentities"
        f"&search={suchbegriff}&language=la&format=json"
    )
    response = requests.get(url, headers={"User-Agent": "CiceroBot/1.0"})
    daten = response.json()
    treffer = daten.get("search", [])
    
    if treffer:
        return treffer[0].get("label", "Unbekannt"), treffer[0]["id"]
    return None, None



In [ ]:
satz = "Marcus Tullius Cicero fuit magnus orator Romanus, sicut eius amicus Atticus."
d = nlp(satz)
html_satz = satz

# Wir extrahieren Namen explizit anhand der BIO-Tags (Begin & Inside)
gefundene_namen = []
aktueller_name = []

for token in d:
    if token.ent_iob_ == "B" and token.ent_type_ in ["PER", "PERSON"]:
        if aktueller_name:
            gefundene_namen.append(" ".join(aktueller_name))
        aktueller_name = [token.text]
    elif token.ent_iob_ == "I" and token.ent_type_ in ["PER", "PERSON"]:
        aktueller_name.append(token.text)
    else:
        if aktueller_name:
            gefundene_namen.append(" ".join(aktueller_name))
            aktueller_name = []
if aktueller_name:
    gefundene_namen.append(" ".join(aktueller_name))



for name in gefundene_namen:
    titel, _id = auto_link_wikidata(name)
    if _id:
        link = f"<a href=\"https://www.wikidata.org/wiki/{_id}\" target=\"_blank\" style=\"color:blue; font-weight:bold;\">{name}</a>"
        html_satz = html_satz.replace(name, f"{link} [Wikidata: {_id}]")

display(HTML("<h4>Verlinkter Historischer Text:</h4>" + html_satz))

### ✍️ Gruppenarbeit 2: Die Live-API loslassen

Iteriert durch alle **Personen** (`PER`) im Beispiel-Brief und ruft für jede die `auto_link_wikipedia()` Funktion auf, um die Wikipedia-Metadaten zu beschaffen.

In [ ]:
# TODO: Euer Code hier!

# 1. Liste aller personen (PER)

# 2. Durchlaufen und die API befragen


## Ausblick: Generative KI und LLMs für NER

Bislang haben wir spezialisierte Modelle (wie `LatinCy`) kennengelernt, die als *Sequence Labeler* jedes Wort prüfen und kategorisieren.

Der neueste Trend beim maschinellen Textverstehen sind **Large Language Models (LLMs)** wie GPT-4, Claude oder Llama. Anstatt komplexe Token-Pipelines und BIO-Tags zu nutzen, geht man bei LLMs über zum **Prompting** (Zero-Shot Information Extraction).

In [ ]:
# Ein Prompt für NER könnte so aussehen:
brief_zitat = "Gaius Iulius Caesar copias ad flumen Rubiconem duxit."

mein_llm_prompt = (
    f"Lies den folgenden lateinischen Text. Extrahiere alle "
    f"geografischen Regionen nach dem Konzept der Pleiades DB und liefere Koordinaten.\n"
    f"Für Personen, liefere links zu Wikidata.\n"
    f"Gib mir das Resultat zwingend als JSON.\n\n"
    f"TEXT: {brief_zitat}"
)

print("Wir senden an ChatGPT/Llama:\n")
print(mein_llm_prompt)

### Diskussion: Vor- und Nachteile von LLMs in den DH

Die Nutzung von LLMs für Named Entity Recognition bietet enorme Chancen, wirft für Historiker aber auch methodische Hürden auf:

**Vorteile:**
- **Zero-Shot:** Wir brauchen keine Tausenden handgeschriebenen Trainingssätze im BIO-Format. Das LLM versteht Anweisungen sofort.
- **Starker Kontext:** Ein LLM löst Ambiguitäten sehr gut auf. Es begreift aus dem Satzbau, ob "Caesar" die Person ist oder ein Buchtitel.

**Nachteile:**
- **Positions-Blindheit:** Historiker wollen oft genau wissen (z.B. für Text Highlighting), an welchem exakten Zeichen das Wort beginnt (`start_char`). LLMs geben meist nur den extrahierten Textteil (als String) zurück.
- **Halluzinationen:** LLMs erfinden (halluzinieren) Eigennamen oder zwingen dem Text Taxonomien auf, die historisch nicht passen.